# TP6 – Inverses modulaires, théorème chinois et RSA

Ce TP met en œuvre les notions vues en cours et en TD sur l'anneau $\mathbb Z/n\mathbb Z$ : éléments inversibles, fonction $\varphi$ d'Euler, congruences linéaires et théorème chinois des restes. On utilise certaines d'elles ensuite pour implémenter le cryptosystème RSA.

**Conventions et fonctions Sage utiles.**

* On peut utiliser les fonctions Sage qui correspondent à ce que l'on a programmé dans les TP précédents : `gcd(a, b)`, `xgcd(a, b)`, `power_mod(a, e, n)`, `is_prime(n)`, `next_prime(n)`, `factor(n)`.
* `random_prime(2**b, proof=False, lbound=2**(b-1))` renvoie un nombre premier aléatoire de $b$ bits, déterminé par un test probabiliste comme celui de Miller–Rabin.
* `randint(a, b)` renvoie un entier aléatoire de $[a, b]$.
* Les fonctions `euler_phi`, `inverse_mod` et `crt` de Sage font ce que l'on demande de programmer dans ce TP : on ne les utilise que pour vérifier nos fonctions et comparer les temps de calcul.
* Lorsqu'une entrée n'est pas valide, une fonction doit **lever une erreur**, par exemple avec `raise ValueError("a n'est pas inversible modulo n")`.

Les cellules *Réponse :* sont à compléter par un texte rédigé.

## 1. Éléments inversibles et fonction $\varphi$ d'Euler

### Exercice 1 — Les éléments inversibles de $\mathbb Z/n\mathbb Z$

1. Écrire une fonction `ElementsInversibles(n)` qui, pour $n \in \mathbb Z_{>0}$, renvoie la liste des entiers $a \in \{0, \dots, n-1\}$ dont la classe est inversible dans $\mathbb Z/n\mathbb Z$, c'est-à-dire l'ensemble $\left(\mathbb Z/n\mathbb Z\right)^{\times}$.
2. Construire la liste des éléments inversibles de $\mathbb Z/323\mathbb Z$. Peut-on déterminer si $323$ est premier, en regardant seulement la longueur de cette liste ?
3. Pourquoi cette méthode n'est-elle pas un bon test de primalité ?

*Réponse :*

### Exercice 2 — Calcul de $\varphi(n)$

On rappelle que la fonction $\varphi$ d'Euler est définie par $\varphi(n) = \#\{a \in \mathbb Z : 1 \le a \le n,\ \operatorname{pgcd}(a, n) = 1\}$. On admet que (exercice du TD5) si $n$ a pour décomposition en facteurs premiers
$$n = p_1^{e_1} p_2^{e_2} \cdots p_r^{e_r},$$
alors
$$\varphi(n) = \left(p_1^{e_1} - p_1^{e_1 - 1}\right)\left(p_2^{e_2} - p_2^{e_2 - 1}\right) \cdots \left(p_r^{e_r} - p_r^{e_r - 1}\right).$$

1. Écrire une fonction `PhiEuler(n)` qui calcule $\varphi(n)$ à l'aide de cette formule.
2. Vérifier, pour $100$ entiers $n$ tirés au hasard entre $1$ et $10^4$, que `PhiEuler(n)`, `len(ElementsInversibles(n))` et `euler_phi(n)` coïncident.
3. Comparer les temps de calcul de `PhiEuler(n)` et de `len(ElementsInversibles(n))` pour des entiers $n$ de plus en plus grands. Quelle est l'étape coûteuse de `PhiEuler` ? 

*Réponse :*

### Exercice 3 — Calcul de l'inverse modulo $n$

1. Écrire une fonction `InverseMod(a, n)` qui calcule l'inverse de $a$ modulo $n$, sous la forme d'un entier de $\{0, \dots, n-1\}$, s'il existe, et qui lève une erreur sinon.
2. Écrire deux autres fonctions qui calculent l'inverse de $a$ modulo $n$ :
   * `InverseModNaif(a, n)`, qui essaie successivement tous les entiers $x \in \{1, \dots, n-1\}$ ;
   * `InverseModEuler(a, n)`, qui utilise le théorème d'Euler : si $\operatorname{pgcd}(a, n) = 1$, alors $a^{\varphi(n)} \equiv 1 \pmod n$.
3. Comparer les temps de calcul de ces trois fonctions et de `inverse_mod(a, n)`, d'abord pour des entiers $n$ de $20$ bits, puis de $1000$ bits. Quelle méthode est utilisable pour de grands entiers $n$ ? Dans quel cas la méthode d'Euler est-elle utilisable ?

## 2. Congruences linéaires

### Exercice 4 — Résoudre une congruence linéaire

On a montré que si $d = \operatorname{pgcd}(a, n)$, la congruence $az \equiv b \pmod n$ admet des solutions si et seulement si $d \mid b$, et que ses solutions sont alors les entiers $z_0 + k\,\frac{n}{d}$, $k \in \mathbb Z$, où $z_0$ est l'une d'entre elles.

1. Écrire une fonction `ResolutionCongruence(a, b, n)` qui renvoie la liste des solutions de $az \equiv b \pmod n$ comprises entre $0$ et $n - 1$, et qui lève une erreur si la congruence n'a pas de solution.
2. Tester la fonction sur les congruences de l'exercice 2 (a) du TD 5.

## 3. Théorème des restes chinois

### Exercice 5 — Le théorème chinois

1. Écrire une fonction `RestesChinois(A, N)` qui prend en entrée deux listes $A = [a_1, \dots, a_k]$ et $N = [n_1, \dots, n_k]$, et qui renvoie l'unique entier $z$ tel que $0 \le z < \prod_i n_i$ et $z \equiv a_i \pmod{n_i}$ pour tout $i \in \{1, \dots, k\}$.

   La fonction doit vérifier que les entrées sont correctes, c'est-à-dire que les deux listes ont la même longueur et que les $n_i$ sont premiers entre eux deux à deux, et lever une erreur sinon.

   *Indication.* Si besoin, lire la documentation des fonctions `prod` et `sum` de Sage pour calculer le produit et la somme des éléments d'une liste.
   
3. Comparer le résultat avec la fonction `crt(A, N)` de Sage sur quelques exemples, dont, par exemple, ceux du TD5.

## 4. Le cryptosystème RSA

RSA (Rivest–Shamir–Adleman, 1977) est un cryptosystème à clé publique encore très utilisé aujourd'hui. Sa sécurité repose sur la difficulté de factoriser de grands entiers, en particulier les produits de deux grands nombres premiers.

Dans sa version de base, RSA comporte trois étapes.

**Génération des clés.**
1. Choisir deux grands nombres premiers distincts $p$ et $q$.
2. Calculer leur produit $N = pq$, appelé *module*.
3. Calculer $\varphi(N) = (p - 1)(q - 1)$.
4. Choisir un exposant $e$ premier avec $\varphi(N)$.
5. Calculer l'inverse $d$ de $e$ modulo $\varphi(N)$.

La clé **publique** est $(N, e)$, la clé **privée** est $d$.

**Chiffrement.** Un message est représenté par un entier $m$ tel que $0 \le m < N$. Son chiffré est $c = m^{e} \bmod N$.

**Déchiffrement.** On retrouve le message à l'aide de la clé privée : $m = c^{d} \bmod N$.

La **sécurité** de RSA repose sur le fait qu'en ne connaissant que $N$ et $e$, il est très difficile de calculer $d$ sans connaître la factorisation $N = pq$.

### Exercice 6 — Pourquoi le déchiffrement fonctionne

On rappelle le *théorème d'Euler* : si $n \ge 1$ et si $a$ est premier avec $n$, alors $a^{\varphi(n)} \equiv 1 \pmod n$.

1. Montrer que si $\operatorname{pgcd}(m, N) = 1$, alors $\left(m^{e}\right)^{d} \equiv m \pmod N$.
2. *(Pour aller plus loin.)* Montrer que le déchiffrement est correct pour **tout** message $0 \le m < N$, même si $\operatorname{pgcd}(m, N) \ne 1$.

   *Indication.* Montrer que $m^{ed} \equiv m \pmod p$ et $m^{ed} \equiv m \pmod q$, puis utiliser le théorème chinois.

*Réponse :*

### Exercice 7 — Génération des clés

Écrire une fonction `RSA_KeyGen(b)` qui prend en entrée un entier $b$ et renvoie un couple de clés publique et privée $(pk, sk)$, selon l'algorithme suivant :
1. tirer au hasard deux nombres premiers distincts $p$ et $q$ de $b$ bits ;
2. calculer $N = pq$ et $\varphi(N) = (p - 1)(q - 1)$ ;
3. tirer au hasard un entier $1 < e < \varphi(N)$ premier avec $\varphi(N)$ ;
4. calculer l'inverse $d$ de $e$ modulo $\varphi(N)$ ;
5. renvoyer $pk = (N, e)$ et $sk = d$.

Générer un couple de clés pour $b = 100$. 

### Exercice 8 — Chiffrement et déchiffrement

1. Écrire une fonction `RSA_Encrypt(m, pk)` qui prend en entrée un entier $0 \le m < N$ et une clé publique $pk = (N, e)$, et renvoie le chiffré $c = m^{e} \bmod N$.
2. Écrire une fonction `RSA_Decrypt(c, pk, sk)` qui prend en entrée un chiffré $c$, la clé publique $pk = (N, e)$ et la clé privée $sk = d$, et renvoie le message $m = c^{d} \bmod N$.
3. Chiffrer un message de votre choix avec la clé de l'exercice 7, puis le déchiffrer, et vérifier qu'on retrouve bien le message de départ.
4. Écrire une fonction de test `RSA_test(b)` qui génère un couple de clés de paramètre $b$, tire au hasard un message $m$, le chiffre puis le déchiffre, et vérifie que l'on retrouve $m$. L'utiliser dans une boucle pour des valeurs de $b$ allant jusqu'à $1000$, et mesurer les temps de génération des clés, de chiffrement et de déchiffrement.

### Exercice 9 — Pourquoi il faut ajouter de l'aléa

Le chiffrement RSA de base est **déterministe** : un même message donne toujours le même chiffré.

1. Mme Iezzi chiffre la note d'un élève sur $20$, un entier $m \in \{0, \dots, 20\}$, avec la clé publique $pk$ de l'université, et envoie le chiffré $c$. Montrer qu'une étudiante qui intercepte $c$ et connaît $pk$ peut retrouver $m$ sans connaître la clé privée. Combien doit-elle calculer de chiffrés pour cette attaque ?
2. Pour éviter cela, on ajoute un **aléa dans le message** avant de le chiffrer. On tire au hasard un entier $r$ de $64$ bits, on chiffre $m' = m \cdot 2^{64} + r$, et, au déchiffrement, on retrouve $m$ comme le quotient de $m'$ par $2^{64}$. Écrire les fonctions de chiffrement et de déchiffrement correspondantes. Quelle condition sur $m$ faut-il imposer ?
3. L'attaque de la question 1 fonctionne-t-elle encore ? Combien de calculs demanderait-elle ?

*Remarque.* En pratique, on utilise des méthodes d'ajout d'aléa plus élaborées, comme OAEP, qui protègent aussi contre d'autres attaques.